## Python libraries for regressions with fixed effect

OLS in statsmodels does not automatically use demeaning or absorb fixed effects when many dummy variables are included as regressors. This is problematic not only for the computational burden (inversion of large matrix) but also for the accuracy in case of near collinearity. It is highly recommended to demean the dataset by the set of dummy variables, which exploits the Frisch-Waugh-Lovell theorem.

### List of options

1. statsmodels.api (**sm**)
2. statsmodels.formula.api (**smf**; accepts Patsy)
3. linearmodels.panel.**PanelOLS** (accepts Formulaic)
4. linearmodels.iv.**AbsorbingLS**
5. **pyfixest** (accepts Formulaic)
6. **reghdfe**

The main takeaways:
- Standard errors from pyfixest was the closest to that obtained from xtreg in Stata.
- Standard errors from reghdfe are sometimes ~10 times larger than the rest.
- For complex combinations of FE, **smf should be avoided** since it may not remove all the collinearities.
- In terms of execution time, **AbsorbingLS, pyfixest, reghdfe** outperformed PanelOLS and sm/smf.
- The current results confirm my personal choice of pyfixest.


### Example exercise

In the code below, I show the result of each of these methods. There are slight differences in the point estimates and the standard errors. I use an example exercise from Meer and West (2016, JHR); the cleaned dataset is available from the author's web site.

Their main regression analysis uses the following specification: 
$$emp_{it} = \beta \cdot mw_{it} + \psi \cdot controls_{it} + \alpha_i + \tau_t [\tau_{rt}] + \gamma_i \cdot t + \epsilon_{it}$$
- $i$ is state in the U.S. (51 including D.C.), $r$ is census region (4; each contains states), and $t$ is year (1977-2011; 35 years).
- $emp_{it}$: log of employment
- $mw_{it}$: log of (real) minimum wage
- $controls_{it}$: log (state population), share of population aged 15-59, log (state level GDP per capita)
- $\alpha_i$, $\tau_t$, $\tau_{rt}$: state FE, year FE, year-region FE
- $\gamma_i \cdot t$: linear time trend for each state

The four specifications are:
1. $emp_{it} = \beta \cdot mw_{it} + \alpha_i + \tau_t + \epsilon_{it}$
2. $emp_{it} = \beta \cdot mw_{it} + \psi \cdot controls_{it} + \alpha_i + \tau_t + \epsilon_{it}$
3. $emp_{it} = \beta \cdot mw_{it} + \psi \cdot controls_{it} + \alpha_i + \tau_{rt} + \epsilon_{it}$
4. $emp_{it} = \beta \cdot mw_{it} + \psi \cdot controls_{it} + \alpha_i + \tau_{rt} + \gamma_i \cdot t + \epsilon_{it}$

Compared to other models with negative and significant impact of $mw_{it}$, model 4 shows much smaller and insignificant impact of $mw_{it}$. The authors interpret this as evidence that the state specific linear time trend is generating a large portion of the seemingly significant impact of $mw_{it}$.


In [40]:

import pandas as pd

df = pd.read_csv('meerwest_bds.csv')
df = df.rename(columns={'statefips': 'state'})
df["_year_region"] = pd.factorize(pd.MultiIndex.from_frame(df[["year", "censusreg"]]))[0]

print(f"""period: from {df['year'].min()} to {df['year'].max()} [{
    df['year'].max() - df['year'].min() + 1} years]""")
print(f"number of states: {df['state'].nunique()}")
print(f"number of regions: {df['censusreg'].nunique()}")
print(f"number of region-year combinations: {df["_year_region"].nunique()}")

controls = ["lpopall", "share1559", "lrgspcap"]

n = 4    # NUMBER OF MODELS
x_list = [[] for _ in range(n)]
fe_list = [[] for _ in range(n)]
# LIST OF FE DUMMY AS DATAFRAME
fd_list = [pd.DataFrame() for _ in range(n)]

d_state = pd.get_dummies(df['state'], drop_first=True, dtype=int)
# REPLACE ' ' BY '_' SINCE SPACES ARE NOT ALLOWED IN PATSY
d_state.columns = [c.replace(' ', '_') for c in d_state.columns]

d_year = pd.get_dummies(df['year'], drop_first=True, dtype=int)
# ADD 'd_' SINCE PURE NUMBERS ARE NOT ALLOWED IN PATSY
d_year.columns = 'd_' + d_year.columns.astype(str)

# FIND THE REGION THAT CONTAINS THE EXCLUDED STATE, AND EXCLUDE THAT REGION
x_region = df.loc[df['state'] == df['state'].drop_duplicates().sort_values()[0], 'censusreg'][0]
a_region = list(df['censusreg'].drop_duplicates().sort_values())
a_region.remove(x_region)
d_region = pd.get_dummies(df['censusreg'], drop_first=False, dtype=int)[a_region]

d_year_region = pd.DataFrame()
for i in range(len(a_region)):
    tmp = d_year.multiply(d_region.iloc[:,i], axis=0)
    tmp.columns = [c + '_' + a_region[i] for c in d_year.columns]
    d_year_region = pd.concat([d_year_region, tmp], axis=1)
display(d_year_region.columns)


for i in range(n):
    x_list[i] = ['lnmin']
    if i >= 1:
        x_list[i] += controls

    if i <= 1:
        fe_list[i] = ["state", "year"]
    else:
        fe_list[i] = ["state", "_year_region"]

    fd_list[i] = pd.concat([d_state, d_year], axis=1)
    if i >= 2:
        fd_list[i] = pd.concat([fd_list[i], d_year_region], axis=1)


assert df.groupby("state")["censusreg"].nunique().max() == 1
t = df["year"] - df["year"].min()
j = 0
x = []
for _, g in df.groupby("censusreg", observed=True):
    for state in pd.unique(g["state"])[1:]:
        name = f"_trend_{j}"
        df[name] = t * df["state"].eq(state)
        x.append(name)
        j += 1
x_list[3] += x

for i in range(n):
    print(i, x_list[i], fe_list[i])

# DATAFRAME AUGMENTED WITH THE FULL SET OF FE DUMMY
df_fd = pd.concat([df, d_state, d_year, d_year_region], axis=1)


# for PanelOLS
p = df.set_index(["state", "year"], drop=False)

# for smf patsy (inaccurate version)
fml_list_patsy = [
    "lemp ~ lnmin + C(state) + C(year)",
    "lemp ~ lnmin + C(state) + C(year) + lpopall + share1559 + lrgspcap",
    "lemp ~ lnmin + C(state) + C(year) * C(censusreg) + lpopall + share1559 + lrgspcap",
    "lemp ~ lnmin + C(state) + C(year) * C(censusreg) + lpopall + share1559 + lrgspcap + year * C(state)",
]

fml_list = [
    "lemp ~ lnmin + ",
    "lemp ~ lnmin + lpopall + share1559 + lrgspcap + ",
    "lemp ~ lnmin + lpopall + share1559 + lrgspcap + ",
    f"lemp ~ {' + '.join(x_list[3])} + ",
]
for i in range(len(fml_list)):
    fml_list[i] += " + ".join(fd_list[i].columns)


period: from 1977 to 2011 [35 years]
number of states: 51
number of regions: 4
number of region-year combinations: 140


Index(['d_1978_Midwest', 'd_1979_Midwest', 'd_1980_Midwest', 'd_1981_Midwest',
       'd_1982_Midwest', 'd_1983_Midwest', 'd_1984_Midwest', 'd_1985_Midwest',
       'd_1986_Midwest', 'd_1987_Midwest',
       ...
       'd_2002_West', 'd_2003_West', 'd_2004_West', 'd_2005_West',
       'd_2006_West', 'd_2007_West', 'd_2008_West', 'd_2009_West',
       'd_2010_West', 'd_2011_West'],
      dtype='str', length=102)

0 ['lnmin'] ['state', 'year']
1 ['lnmin', 'lpopall', 'share1559', 'lrgspcap'] ['state', 'year']
2 ['lnmin', 'lpopall', 'share1559', 'lrgspcap'] ['state', '_year_region']
3 ['lnmin', 'lpopall', 'share1559', 'lrgspcap', '_trend_0', '_trend_1', '_trend_2', '_trend_3', '_trend_4', '_trend_5', '_trend_6', '_trend_7', '_trend_8', '_trend_9', '_trend_10', '_trend_11', '_trend_12', '_trend_13', '_trend_14', '_trend_15', '_trend_16', '_trend_17', '_trend_18', '_trend_19', '_trend_20', '_trend_21', '_trend_22', '_trend_23', '_trend_24', '_trend_25', '_trend_26', '_trend_27', '_trend_28', '_trend_29', '_trend_30', '_trend_31', '_trend_32', '_trend_33', '_trend_34', '_trend_35', '_trend_36', '_trend_37', '_trend_38', '_trend_39', '_trend_40', '_trend_41', '_trend_42', '_trend_43', '_trend_44', '_trend_45', '_trend_46'] ['state', '_year_region']


In [41]:

import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from linearmodels.iv import AbsorbingLS
import pyfixest as pf
from reghdfe import reghdfe

import time
import warnings

lib_list = ['sm', 'smf', 'smf_x', 'panel', 'absorbing', 'pyfixest', 'reghdfe']
summaries = [{} for _ in range(2)]
models = [{} for _ in range(2)]
runtimes = [{} for _ in range(2)]

for lib in lib_list:

    # j = 0 FOR DEFAULT STDERR, 1 FOR CLUSTERED STDERR
    for j in range(2):
        summaries[j][lib] = []
        models[j][lib] = []
        runtimes[j][lib] = []

        for i in range(n):

            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                start = time.perf_counter()

                # DEFINE ARGUMENT FOR STDERR AND RUN REGRESSION

                if lib == 'sm':
                    args_fit = {'cov_type': "cluster", 'cov_kwds': {"groups": df["state"]}} if j == 1 else {}

                    result = sm.OLS(
                        df['lemp'],
                        pd.concat([
                            sm.add_constant(df[x_list[i]]),
                            fd_list[i],
                        ], axis=1)
                    ).fit(**args_fit)

                elif lib in ['smf', 'smf_x']:
                    args_fit = {'cov_type': "cluster", 'cov_kwds': {"groups": df["state"]}} if j == 1 else {}
                    fml = fml_list[i] if lib == 'smf' else fml_list_patsy[i]

                    result = smf.ols(
                        formula=fml,
                        data=df_fd,
                    ).fit(**args_fit)

                elif lib == 'panel':
                    kwargs = {"time_effects": True} if i <= 1 else {"other_effects": p[["_year_region"]]}
                    args_fit = {'cov_type': "clustered", 'cluster_entity': True} if j == 1 else {}
                    
                    result = PanelOLS(
                        p["lemp"],
                        p[x_list[i]],
                        entity_effects=True,
                        drop_absorbed=True,
                        **kwargs,
                    ).fit(**args_fit)

                elif lib == 'absorbing':
                    args_fit = {'cov_type': "clustered", 'clusters': df["state"], 'debiased': True} if j == 1 else {}

                    result = AbsorbingLS(
                        df["lemp"],
                        df[x_list[i]],
                        absorb=df[fe_list[i]].astype("category"),
                        drop_absorbed=True,
                    ).fit(**args_fit)

                    summary = pd.DataFrame({
                        "coef": result.params,
                        "se": result.std_errors,
                        "pvalue": result.pvalues,
                    }).loc[["lnmin"]]
                    summary[['n', 'k']] = [result.nobs, len(result.params)]

                elif lib == 'pyfixest':
                    fml = f"lemp ~ {' + '.join(x_list[i])} | {' + '.join(fe_list[i])}"
                    args_fit = {'vcov': {"CRV1": "state"}} if j == 1 else {}
                    
                    result = pf.feols(
                        fml,
                        data=df,
                        **args_fit,
                        fixef_rm="none",
                    )
                    
                elif lib == 'reghdfe':
                    args_fit = {'cov_type': "cluster", 'cluster': ["state"]} if j == 1 else {}

                    result = reghdfe(
                        data=df,
                        y="lemp",
                        x=x_list[i],
                        fe=fe_list[i],
                        **args_fit,
                        drop_singletons=False,
                    )

                # COLLECT RESULTS FROM THE REGRESSION

                if lib in ['sm', 'smf', 'smf_x', 'reghdfe']:
                    summary = pd.DataFrame({
                        "coef": result.params,
                        "se": result.bse,
                        "pvalue": result.pvalues,
                    }).loc[["lnmin"]]
                    summary[['n', 'k']] = [result.nobs, len(result.params)]

                elif lib in ['panel', 'absorbing']:
                    summary = pd.DataFrame({
                        "coef": result.params,
                        "se": result.std_errors,
                        "pvalue": result.pvalues,
                    }).loc[["lnmin"]]
                    summary[['n', 'k']] = [result.nobs, len(result.params)]

                elif lib in ['pyfixest']:                    
                    summary = pd.DataFrame({
                        "coef": result.coef(),
                        "se": result.se(),
                        "pvalue": result.pvalue(),
                    }).loc[["lnmin"]]
                    summary[['n', 'k']] = [result._N, result._k]
                    

                summary['warning'] = 0
                summary['warning_category'] = [[]]
                summary['warning_message'] = [[]]
                if len(w) > 0:
                    #summary['warning_category'] += [str(w[0].category).split('.')[-1][:-2]]
                    for warning in w:
                        # reghdfe always returns this warning (not informative)
                        if str(warning.category) != "<class 'pandas.errors.Pandas4Warning'>":
                            summary['warning'] += 1
                            summary['warning_category'] += [[str(warning.category)]]
                            summary['warning_message'] += [[str(warning.message)]]
                        
                summaries[j][lib].append(summary)
                models[j][lib].append(result)
                runtimes[j][lib].append(time.perf_counter() - start)

runtimes_sum = {}
for lib in lib_list:
    runtimes_sum[lib] = 0

    for j in range(2):
        runtimes_sum[lib] += sum(runtimes[j][lib])

display(pd.DataFrame(runtimes_sum, index=[0]).T)


summary_all = pd.DataFrame()
for i, x in enumerate(lib_list):
    for j in range(2):
        tab = pd.concat(
            summaries[j][x],
            axis=0,
            keys=[(f"Model_{k+1}", f"{j}", f"{i+1}_{x}") for k in range(n)],
            names=["model", "cluster", "library", "regressor"],
        )
        summary_all = pd.concat([summary_all, tab])

summary_all = summary_all.sort_index().reset_index().drop(columns=['regressor'])
print(summary_all)
#summary_all.to_csv('reg_fe_test.csv', index=False)


,0
sm,0.358109
smf,1.705451
smf_x,0.658107
panel,0.549830
absorbing,0.167125
pyfixest,0.192263
reghdfe,0.117606


      model cluster      library      coef        se        pvalue       n  \
0   Model_1       0         1_sm -0.185940  0.042053  1.042041e-05  1785.0   
1   Model_1       0        2_smf -0.185940  0.042053  1.042041e-05  1785.0   
2   Model_1       0      3_smf_x -0.185940  0.042053  1.042041e-05  1785.0   
3   Model_1       0      4_panel -0.185940  0.042053  1.042041e-05  1785.0   
4   Model_1       0  5_absorbing -0.185940  0.042474  1.199268e-05  1785.0   
5   Model_1       0   6_pyfixest -0.185940  0.042053  1.042041e-05  1785.0   
6   Model_1       0    7_reghdfe -0.185940  0.042486  1.279435e-05  1785.0   
7   Model_1       1         1_sm -0.185940  0.119480  1.196520e-01  1785.0   
8   Model_1       1        2_smf -0.185940  0.119480  1.196520e-01  1785.0   
9   Model_1       1      3_smf_x -0.185940  0.119480  1.196520e-01  1785.0   
10  Model_1       1      4_panel -0.185940  0.118336  1.163036e-01  1785.0   
11  Model_1       1  5_absorbing -0.185940  0.116599  1.109677e-


It turns out that "smf_x" raises collinearity warning (rank-deficient), and the results are slighlty different from "sm" or "smf". Despite the exact collinearity, the results are not too different, but it is clearly not the best practice. Note that the number of regressor in "smf_x" exceeds that of "sm" or "smf".

The default standard errors (j=0) from absorbing and reghdfe are different from the rest. This is because the default is set to robust standard errors in these methods.


### Patsy vs. Formulaic

Formulaic is a high-performance, extensible successor to patsy designed to overcome the older library's speed and architectural limitations.

While patsy has historically been the default formula engine for statistical libraries like statsmodels, major frameworks have largely moved to formulaic as the modern standard.

In [43]:

dfx = df.rename(columns={"lemp": "y", "lnmin": "x1", "lpopall": "x2", "state": "group"})[["y", "x1", "x2", "group"]]

from patsy import dmatrix, dmatrices
X = dmatrix("x1 + x2", data=dfx, return_type="dataframe")
y, X = dmatrices("y ~ x1 + x2 + C(group)", data=dfx, return_type="dataframe")
y, X = dmatrices("y ~ x1 + x2 + C(group, Treatment(reference='Alabama'))", data=dfx, return_type="dataframe")
# return_type="dataframe" <class 'pandas.DataFrame'>
# return_type="matrix" <class 'patsy.design_info.DesignMatrix'> [default]
print(type(X))
display(X)

from formulaic import model_matrix
X = model_matrix("x1 + x2", dfx, output="pandas")
y, X = model_matrix("y ~ x1 + x2 + C(group)", dfx, output="pandas")
y, X = model_matrix("y ~ x1 + x2 + C(group, contr.treatment(base='Alabama'))", dfx, output="pandas")
# output="pandas" <class 'formulaic.model_matrix.ModelMatrix'> [default]
print(type(X))
display(X)


<class 'pandas.DataFrame'>


,Intercept,"C(group, Treatment(reference='Alabama'))[T.Alaska]","C(group, Treatment(reference='Alabama'))[T.Arizona]","C(group, Treatment(reference='Alabama'))[T.Arkansas]","C(group, Treatment(reference='Alabama'))[T.California]","C(group, Treatment(reference='Alabama'))[T.Colorado]","C(group, Treatment(reference='Alabama'))[T.Connecticut]","C(group, Treatment(reference='Alabama'))[T.Delaware]","C(group, Treatment(reference='Alabama'))[T.District of Columbia]","C(group, Treatment(reference='Alabama'))[T.Florida]",...,"C(group, Treatment(reference='Alabama'))[T.Texas]","C(group, Treatment(reference='Alabama'))[T.Utah]","C(group, Treatment(reference='Alabama'))[T.Vermont]","C(group, Treatment(reference='Alabama'))[T.Virginia]","C(group, Treatment(reference='Alabama'))[T.Washington]","C(group, Treatment(reference='Alabama'))[T.West Virginia]","C(group, Treatment(reference='Alabama'))[T.Wisconsin]","C(group, Treatment(reference='Alabama'))[T.Wyoming]",x1,x2
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.188106,15.145914
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.123246,15.159411
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.178556,15.168594
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.143892,15.176581
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.092655,15.181228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1780,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.750802,13.189790
1781,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.710800,13.210453
1782,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.837331,13.235426
1783,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.923505,13.237433


<class 'formulaic.model_matrix.ModelMatrix'>


,Intercept,x1,x2,"C(group, contr.treatment(base='Alabama'))[T.Alaska]","C(group, contr.treatment(base='Alabama'))[T.Arizona]","C(group, contr.treatment(base='Alabama'))[T.Arkansas]","C(group, contr.treatment(base='Alabama'))[T.California]","C(group, contr.treatment(base='Alabama'))[T.Colorado]","C(group, contr.treatment(base='Alabama'))[T.Connecticut]","C(group, contr.treatment(base='Alabama'))[T.Delaware]",...,"C(group, contr.treatment(base='Alabama'))[T.South Dakota]","C(group, contr.treatment(base='Alabama'))[T.Tennessee]","C(group, contr.treatment(base='Alabama'))[T.Texas]","C(group, contr.treatment(base='Alabama'))[T.Utah]","C(group, contr.treatment(base='Alabama'))[T.Vermont]","C(group, contr.treatment(base='Alabama'))[T.Virginia]","C(group, contr.treatment(base='Alabama'))[T.Washington]","C(group, contr.treatment(base='Alabama'))[T.West Virginia]","C(group, contr.treatment(base='Alabama'))[T.Wisconsin]","C(group, contr.treatment(base='Alabama'))[T.Wyoming]"
0,1.0,2.188106,15.145914,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1.0,2.123246,15.159411,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1.0,2.178556,15.168594,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.0,2.143892,15.176581,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1.0,2.092655,15.181228,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1780,1.0,1.750802,13.189790,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1781,1.0,1.710800,13.210453,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1782,1.0,1.837331,13.235426,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1783,1.0,1.923505,13.237433,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
